In [1]:
## 1. Import Libraries and Initialize Google Earth Engine
import ee
import geemap
import geopandas as gpd
from pathlib import Path

In [2]:
### Earth Engine Authentication and Initialization
ee.Authenticate()
ee.Initialize()

In [4]:
### Load the Colombian Municipal Boundaries
root_folder = Path(r"C:\Users\CAMILO\Downloads")

gdf = gpd.read_file(
    root_folder / "municipios_colombia.gpkg"
)

print("CRS original:", gdf.crs)
print(gdf.columns)

CRS original: EPSG:3116
Index(['DPTO_CCDGO', 'MPIO_CCDGO', 'MPIO_CNMBR', 'MPIO_CDPMP', 'VERSION',
       'AREA', 'LATITUD', 'LONGITUD', 'STCTNENCUE', 'STP3_1_SI', 'STP3_2_NO',
       'STP3A_RI', 'STP3B_TCN', 'STP4_1_SI', 'STP4_2_NO', 'STP9_1_USO',
       'STP9_2_USO', 'STP9_3_USO', 'STP9_4_USO', 'STP9_2_1_M', 'STP9_2_2_M',
       'STP9_2_3_M', 'STP9_2_4_M', 'STP9_2_9_M', 'STP9_3_1_N', 'STP9_3_2_N',
       'STP9_3_3_N', 'STP9_3_4_N', 'STP9_3_5_N', 'STP9_3_6_N', 'STP9_3_7_N',
       'STP9_3_8_N', 'STP9_3_9_N', 'STP9_3_10', 'STP9_3_99', 'STVIVIENDA',
       'STP14_1_TI', 'STP14_2_TI', 'STP14_3_TI', 'STP14_4_TI', 'STP14_5_TI',
       'STP14_6_TI', 'STP15_1_OC', 'STP15_2_OC', 'STP15_3_OC', 'STP15_4_OC',
       'TSP16_HOG', 'STP19_EC_1', 'STP19_ES_2', 'STP19_EE_1', 'STP19_EE_2',
       'STP19_EE_3', 'STP19_EE_4', 'STP19_EE_5', 'STP19_EE_6', 'STP19_EE_9',
       'STP19_ACU1', 'STP19_ACU2', 'STP19_ALC1', 'STP19_ALC2', 'STP19_GAS1',
       'STP19_GAS2', 'STP19_GAS9', 'STP19_REC1', 'STP19_REC2',

In [5]:
### Reproject the Study Area to EPSG:9377
gdf_unico = gdf.to_crs(epsg=9377)

In [8]:
### Select the Municipality of Interest
gdf_muni = gdf_unico[
    gdf_unico["MPIO_CNMBR"] == "LA PLATA"
].copy()

In [9]:
### Convert the Study Area to WGS84 for Google Earth Engine
gdf_muni_4326 = gdf_muni.to_crs(epsg=4326)

ee_muni_geom = ee.Geometry(
    gdf_muni_4326.geometry.iloc[0].__geo_interface__
)

In [10]:
### Calculate the Map Center
centroid = [
    gdf_muni_4326.geometry.iloc[0].centroid.y,
    gdf_muni_4326.geometry.iloc[0].centroid.x
]

print(centroid)

[2.328076586481396, -76.00117298752485]


In [ ]:
## 2. Sentinel-2 EVI Annual Composites

##The Enhanced Vegetation Index (EVI) was selected as the alternative spectral index because it is sensitive to vegetation canopy conditions while reducing some of the influence of atmospheric effects and soil background compared with NDVI.

##EVI was calculated using the NIR, red and blue spectral bands.

In [12]:

def mask_s2_clouds_scl(image):
    scl = image.select("SCL")
    
    mask = (
        scl.neq(3)   # Cloud shadow
        .And(scl.neq(8))   # Cloud medium probability
        .And(scl.neq(9))   # Cloud high probability
        .And(scl.neq(10))  # Cirrus
        .And(scl.neq(11))  # Snow/ice
    )
    
    return image.updateMask(mask)

In [17]:

years = list(range(2017, 2026))

for year in years:
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(
            f"{year}-01-01",
            f"{year + 1}-01-01"
        )
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                50
            )
        )
    )
    
    print(
        year,
        collection.size().getInfo()
    )

2017 2
2018 5
2019 20
2020 27
2021 12
2022 19
2023 12
2024 21
2025 21


In [18]:
def compute_annual_evi_s2(year):
    
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year + 1, 1, 1)
    
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                50
            )
        )
        .map(mask_s2_clouds_scl)
    )
    
    composite = (
        collection
        .median()
        .clip(ee_muni_geom)
    )
    
    evi = (
        composite
        .expression(
            "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
            {
                "NIR": composite.select("B8"),
                "RED": composite.select("B4"),
                "BLUE": composite.select("B2")
            }
        )
        .rename(f"EVI_{year}")
    )
    
    return evi.set("year", year)

In [19]:
annual_evi_s2 = ee.ImageCollection.fromImages(
    [
        compute_annual_evi_s2(year)
        for year in years
    ]
)
print(
    annual_evi_s2.size().getInfo()
)

9


In [20]:
evi_2023_s2 = (
    annual_evi_s2
    .filter(ee.Filter.eq("year", 2023))
    .first()
)

Map_EVI = geemap.Map(
    center=centroid,
    zoom=11
)

Map_EVI.addLayer(
    evi_2023_s2,
    {
        "min": 0,
        "max": 0.8,
        "palette": [
            "brown",
            "yellow",
            "green",
            "darkgreen"
        ]
    },
    "Sentinel-2 EVI 2023"
)

Map_EVI

Map(center=[2.328076586481396, -76.00117298752485], controls=(WidgetControl(options=['position', 'transparent_…

In [21]:
def mask_landsat8(image):
    
    qa = image.select("QA_PIXEL")
    
    mask = (
        qa.bitwiseAnd(1 << 1).eq(0)   # Dilated cloud
        .And(qa.bitwiseAnd(1 << 2).eq(0))  # Cirrus
        .And(qa.bitwiseAnd(1 << 3).eq(0))  # Cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # Cloud shadow
        .And(qa.bitwiseAnd(1 << 5).eq(0))  # Snow
    )
    
    return image.updateMask(mask)

In [22]:
def scale_landsat8(image):
    
    optical_bands = (
        image
        .select("SR_B.")
        .multiply(0.0000275)
        .add(-0.2)
    )
    
    return image.addBands(
        optical_bands,
        None,
        True
    )

In [23]:
def compute_annual_evi_l8(year):
    
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year + 1, 1, 1)
    
    collection = (
        ee.ImageCollection(
            "LANDSAT/LC08/C02/T1_L2"
        )
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(
            ee.Filter.lt(
                "CLOUD_COVER",
                50
            )
        )
        .map(mask_landsat8)
        .map(scale_landsat8)
    )
    
    composite = (
        collection
        .median()
        .clip(ee_muni_geom)
    )
    
    evi = (
        composite
        .expression(
            "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
            {
                "NIR": composite.select("SR_B5"),
                "RED": composite.select("SR_B4"),
                "BLUE": composite.select("SR_B2")
            }
        )
        .rename(f"EVI_{year}")
    )
    
    return evi.set("year", year)

In [24]:
annual_evi_l8 = ee.ImageCollection.fromImages(
    [
        compute_annual_evi_l8(year)
        for year in years
    ]
)
print(
    annual_evi_l8.size().getInfo()
)

9


In [25]:
s2_stack = annual_evi_s2.toBands()
print(
    s2_stack.bandNames().getInfo()
)

['0_EVI_2017', '1_EVI_2018', '2_EVI_2019', '3_EVI_2020', '4_EVI_2021', '5_EVI_2022', '6_EVI_2023', '7_EVI_2024', '8_EVI_2025']


In [26]:
s2_stack = s2_stack.rename(
    [f"EVI_{year}" for year in years]
)
print(
    s2_stack.bandNames().getInfo()
)

['EVI_2017', 'EVI_2018', 'EVI_2019', 'EVI_2020', 'EVI_2021', 'EVI_2022', 'EVI_2023', 'EVI_2024', 'EVI_2025']


In [27]:
l8_stack = annual_evi_l8.toBands()

l8_stack = l8_stack.rename(
    [f"EVI_{year}" for year in years]
)
print(
    l8_stack.bandNames().getInfo()
)

['EVI_2017', 'EVI_2018', 'EVI_2019', 'EVI_2020', 'EVI_2021', 'EVI_2022', 'EVI_2023', 'EVI_2024', 'EVI_2025']


In [28]:
task_s2 = ee.batch.Export.image.toDrive(
    image=s2_stack,
    description="S2_EVI_Stack_2016_2025",
    folder="GEE_Final_Assessment",
    fileNamePrefix="S2_EVI_Stack_2016_2025",
    region=ee_muni_geom,
    scale=10,
    crs="EPSG:9377",
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

task_s2.start()

print("Exportación Sentinel-2 iniciada")

Exportación Sentinel-2 iniciada


In [29]:
task_l8 = ee.batch.Export.image.toDrive(
    image=l8_stack,
    description="L8_EVI_Stack_2016_2025",
    folder="GEE_Final_Assessment",
    fileNamePrefix="L8_EVI_Stack_2016_2025",
    region=ee_muni_geom,
    scale=30,
    crs="EPSG:9377",
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

task_l8.start()

print("Exportación Landsat 8 iniciada")

Exportación Landsat 8 iniciada


In [30]:
print(
    ee.batch.Task.list()
)

[<Task YE4QTCQJT336DP6K5YH43BSY EXPORT_IMAGE: L8_EVI_Stack_2016_2025 (FAILED)>, <Task ZBW6YPJ2QCWPBYS6SI4CGOYB EXPORT_IMAGE: S2_EVI_Stack_2016_2025 (FAILED)>]


In [31]:
Map_stack = geemap.Map(
    center=centroid,
    zoom=11
)

Map_stack.addLayer(
    s2_stack.select("EVI_2023"),
    {
        "min": 0,
        "max": 0.8,
        "palette": [
            "brown",
            "yellow",
            "green",
            "darkgreen"
        ]
    },
    "Sentinel-2 EVI 2023"
)

Map_stack.addLayer(
    l8_stack.select("EVI_2023"),
    {
        "min": 0,
        "max": 0.8,
        "palette": [
            "brown",
            "yellow",
            "green",
            "darkgreen"
        ]
    },
    "Landsat 8 EVI 2023",
    False
)

Map_stack.add_layer_control()

Map_stack

Map(center=[2.328076586481396, -76.00117298752485], controls=(WidgetControl(options=['position', 'transparent_…